In [ ]:
import pandas as pd
from pandas.errors import PerformanceWarning
import warnings

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

warnings.simplefilter('ignore', category=PerformanceWarning)

In [ ]:
from src.features.load_intra_day_dataset import load_intra_day_dataset

start = pd.Timestamp("2024-07-01")
end = pd.Timestamp("2025-04-01")
val_cutoff = pd.Timestamp("2025-01-01")
select_cutoff = pd.Timestamp("2025-02-01")
test_cutoff = pd.Timestamp("2025-03-01")

df = load_intra_day_dataset(start, end)

In [ ]:
from src.models.data_split import tabular_split, time_series_split, ID_KNOWN
from src.models.train_intra_day_time_series import train_intra_day_time_series
from src.models.train_intra_day_tabular import train_intra_day_tabular
from src.models.evaluation import evaluate_timeseries_model, evaluate_tabular_model
from src.models.metrics import get_timeseries_metrics, get_tabular_metrics

tab_df = df[[col for col in df.columns if any(col.startswith(prefix) for prefix in ID_KNOWN)] + ["ID Weighted average of the prices (EUR/MWh)"]]
tab_train, tab_val, tab_select, tab_test = tabular_split(tab_df, val_cutoff, select_cutoff, test_cutoff)
ts_train, ts_val, ts_select, ts_test = time_series_split(df, val_cutoff, select_cutoff, test_cutoff)
ts_metrics = get_tabular_metrics() # TODO: Metrics - Fix
tab_metrics = get_tabular_metrics()

In [ ]:
ts_mae_model = train_intra_day_time_series(ts_train, ts_val, "MAE", cache="MAE")
display(evaluate_timeseries_model(ts_mae_model, ts_train, ts_val, ts_select, ts_metrics))

In [ ]:
tab_mae_model = train_intra_day_tabular(tab_train, tab_val, "MAE", cache="MAE")
display(evaluate_tabular_model(tab_mae_model, tab_train, tab_val, tab_select, tab_metrics))

In [ ]:
# from autogluon.timeseries import TimeSeriesDataFrame
#
# ts_retrain = TimeSeriesDataFrame.from_data_frame(pd.concat([ts_train, ts_val, ts_select]))
# ts_model = train_intra_day_time_series(ts_retrain, None, "MAE", cache="retrain", models={"Chronos": {}})
# display(ts_model.leaderboard(ts_test, extra_metrics=ts_metrics))